# ДЗ-1. Разведочный анализ данных (EDA)

**Курс:** нулевой модуль магистратуры «Искусственный интеллект», НИУ ВШЭ ФКН.

**Датасет:** «Перечень субсидий, предоставленных организациям, осуществляющим
деятельность в инновационной и промышленной сферах» — набор № 591 портала открытых
данных Правительства Москвы (<https://data.mos.ru/opendata/591>).
Поставщик данных — Департамент инвестиционной и промышленной политики города Москвы.
Используется выгрузка в формате XLSX, версия от 17.04.2018.

**Что делает ноутбук:**

1. загружает исходную выгрузку без правок;
2. приводит ее к пригодному для анализа виду и фиксирует, что именно пришлось починить;
3. проверяет данные на пропуски, явные и неявные дубликаты, аномальные значения;
4. собирает профилирующий отчет `report.html` библиотекой `ydata-profiling`.

In [1]:
from pathlib import Path

import pandas as pd
from ydata_profiling import ProfileReport

HERE = Path.cwd()
DATA = HERE / "data" / "data-591-2018-04-17.xlsx"
REPORT = HERE / "report.html"

# Столбцы, которые по смыслу числовые, хотя в выгрузке лежат текстом
NUMERIC_COLS = {
    "Сумма субсидии по договору (руб.)": "int64",
    "Начало действия договора субсидии": "int64",
    "Конец действия договора субсидии": "int64",
}

pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 60)

/Users/iamartempn/Education/HSE/courses/module0/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/var/folders/2l/sxyr8tx14jz3jyx_9kzv4wkh0000gn/T/ipykernel_4826/1469090865.py:4: DeprecationWarning: 
    `import ydata_profiling` is deprecated and will not receive more updates. 
    Please install fg-data-profiling via `pip install fg-data-profiling` and use `import data_profiling` instead.
    
  from ydata_profiling import ProfileReport


## 1. Загрузка исходных данных

In [2]:
raw = pd.read_excel(DATA)

print("Размерность выгрузки:", raw.shape)
raw.head(3)

Размерность выгрузки: (271, 7)


/Users/iamartempn/Education/HSE/courses/module0/.venv/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,ID,Name,SubsidyRecipient,global_id,StartDate,EndDate,SubsidySum
0,Код,Наименование,Получатель субсидии,global_id,Начало действия договора субсидии,Конец действия договора субсидии,Сумма субсидии по договору (руб.)
1,86,Техническое перевооружение производственного комплекса О...,Открытое акционерное общество «Царицыно»,4712493,2012,2015,23520879
2,23,Выход на РИИ ММВБ ОАО «СкайКомпьютинг»,Открытое акционерное общество «СкайКомпьютинг»,4712509,2012,2016,5000000


Первая строка выгрузки — не наблюдение, а второй ряд заголовков: технические имена
колонок (`Name`, `SubsidySum`) продублированы русскими подписями. Из-за этого `pandas`
считает все столбцы текстовыми, включая суммы и годы.

In [3]:
# Русские подписи из служебной строки забираем как имена колонок, саму строку убираем
labels = raw.iloc[0].to_dict()
print("Служебная строка с подписями:")
for tech, rus in labels.items():
    print(f"  {tech:<18} -> {rus}")

print("\nТипы столбцов до обработки:")
print(raw.dtypes)

Служебная строка с подписями:
  ID                 -> Код
  Name               -> Наименование
  SubsidyRecipient   -> Получатель субсидии
  global_id          -> global_id
  StartDate          -> Начало действия договора субсидии
  EndDate            -> Конец действия договора субсидии
  SubsidySum         -> Сумма субсидии по договору (руб.)

Типы столбцов до обработки:
ID                  object
Name                object
SubsidyRecipient    object
global_id           object
StartDate           object
EndDate             object
SubsidySum          object
dtype: object


In [4]:
df = raw.iloc[1:].rename(columns=labels).reset_index(drop=True)
df.columns = [str(c).strip() for c in df.columns]

print("Размерность после снятия служебной строки:", df.shape)
print("Столбцы:", list(df.columns))
df.head(3)

Размерность после снятия служебной строки: (270, 7)
Столбцы: ['Код', 'Наименование', 'Получатель субсидии', 'global_id', 'Начало действия договора субсидии', 'Конец действия договора субсидии', 'Сумма субсидии по договору (руб.)']


,Код,Наименование,Получатель субсидии,global_id,Начало действия договора субсидии,Конец действия договора субсидии,Сумма субсидии по договору (руб.)
0,86,Техническое перевооружение производственного комплекса О...,Открытое акционерное общество «Царицыно»,4712493,2012,2015,23520879
1,23,Выход на РИИ ММВБ ОАО «СкайКомпьютинг»,Открытое акционерное общество «СкайКомпьютинг»,4712509,2012,2016,5000000
2,24,Выход на РИИ ММВБ ОАО «Мультисистема»,Открытое акционерное общество «Мультисистема»,4712510,2012,2016,5000000


## 2. Первичный осмотр

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 270 entries, 0 to 269
Data columns (total 7 columns):
 #   Column                             Non-Null Count  Dtype 
---  ------                             --------------  ----- 
 0   Код                                270 non-null    object
 1   Наименование                       270 non-null    object
 2   Получатель субсидии                270 non-null    object
 3   global_id                          270 non-null    object
 4   Начало действия договора субсидии  270 non-null    object
 5   Конец действия договора субсидии   270 non-null    object
 6   Сумма субсидии по договору (руб.)  270 non-null    object
dtypes: object(7)
memory usage: 14.9+ KB


In [6]:
print("Уникальных значений в каждом столбце:")
print(df.nunique().sort_values(ascending=False))

Уникальных значений в каждом столбце:
Код                                  270
global_id                            270
Наименование                         268
Получатель субсидии                  241
Сумма субсидии по договору (руб.)    223
Конец действия договора субсидии      10
Начало действия договора субсидии      6
dtype: int64


## 3. Проверка качества данных

### 3.1. Пропуски и явные дубликаты

In [7]:
gaps = pd.DataFrame(
    {
        "пропусков": df.isna().sum(),
        "доля, %": (df.isna().mean() * 100).round(1),
        "пустых строк": df.apply(lambda s: s.astype(str).str.strip().eq("").sum()),
    }
)
gaps

,пропусков,"доля, %",пустых строк
Код,0,0.0,0
Наименование,0,0.0,0
Получатель субсидии,0,0.0,0
global_id,0,0.0,0
Начало действия договора субсидии,0,0.0,0
Конец действия договора субсидии,0,0.0,0
Сумма субсидии по договору (руб.),0,0.0,0


In [8]:
key = ["Наименование", "Получатель субсидии", "Сумма субсидии по договору (руб.)"]

print("Полных дубликатов строк:", df.duplicated().sum())
print("Дубликатов по смысловому ключу (наименование + получатель + сумма):",
      df.duplicated(subset=key).sum())
print("Дубликатов идентификатора global_id:", df["global_id"].duplicated().sum())
print("Дубликатов кода:", df["Код"].duplicated().sum())

Полных дубликатов строк: 0
Дубликатов по смысловому ключу (наименование + получатель + сумма): 0
Дубликатов идентификатора global_id: 0
Дубликатов кода: 0


### 3.2. Мусорные символы в текстовых полях

In [9]:
text_cols = [c for c in df.columns if c not in NUMERIC_COLS]

dirt = pd.DataFrame(
    {
        "перенос строки внутри значения": [
            df[c].astype(str).str.contains("\n").sum() for c in text_cols
        ],
        "лишние пробелы по краям": [
            (df[c].astype(str).str.strip() != df[c].astype(str)).sum() for c in text_cols
        ],
        "двойные пробелы внутри": [
            df[c].astype(str).str.contains("  ").sum() for c in text_cols
        ],
    },
    index=text_cols,
)
dirt

,перенос строки внутри значения,лишние пробелы по краям,двойные пробелы внутри
Код,0,0,0
Наименование,48,40,0
Получатель субсидии,35,25,0
global_id,0,0,0


### 3.3. Приведение типов

Суммы и годы хранятся текстом. Проверяем, что все значения разбираются как числа,
и только после этого меняем тип — иначе `errors="coerce"` тихо превратит брак в пропуски.

In [10]:
for col in NUMERIC_COLS:
    broken = pd.to_numeric(df[col], errors="coerce").isna().sum()
    print(f"{col}: значений, не разобранных как число — {broken}")

clean = df.copy()

# Сначала чистим текст, потом меняем тип: обратный порядок снова сделал бы суммы строками
for col in text_cols:
    clean[col] = clean[col].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()

for col, dtype in NUMERIC_COLS.items():
    clean[col] = pd.to_numeric(clean[col]).astype(dtype)

clean.dtypes

Сумма субсидии по договору (руб.): значений, не разобранных как число — 0
Начало действия договора субсидии: значений, не разобранных как число — 0
Конец действия договора субсидии: значений, не разобранных как число — 0


Код                                  object
Наименование                         object
Получатель субсидии                  object
global_id                            object
Начало действия договора субсидии     int64
Конец действия договора субсидии      int64
Сумма субсидии по договору (руб.)     int64
dtype: object

### 3.4. Аномалии в суммах и сроках

In [11]:
summa = clean["Сумма субсидии по договору (руб.)"]

print("Описательные статистики по сумме субсидии, руб.:")
print(summa.describe().apply(lambda v: f"{v:,.0f}".replace(",", " ")))
print("\nКвантили 0.9 / 0.95 / 0.99:")
print(summa.quantile([0.9, 0.95, 0.99]).apply(lambda v: f"{v:,.0f}".replace(",", " ")))
print("\nСуммы меньше 1000 руб.:", sorted(summa[summa < 1000].tolist()))

Описательные статистики по сумме субсидии, руб.:
count            270
mean      43 102 426
std      116 233 421
min                1
25%        5 077 830
50%        7 000 000
75%       21 761 108
max      865 000 000
Name: Сумма субсидии по договору (руб.), dtype: object

Квантили 0.9 / 0.95 / 0.99:
0.90     94 060 240
0.95    194 577 166
0.99    642 298 642
Name: Сумма субсидии по договору (руб.), dtype: object

Суммы меньше 1000 руб.: [1, 3]


In [12]:
clean.loc[summa < 1000, ["Наименование", "Получатель субсидии",
                          "Сумма субсидии по договору (руб.)"]]

,Наименование,Получатель субсидии,Сумма субсидии по договору (руб.)
229,Возмещение части фактически произведенных затрат на упла...,АО «КОРПОРАЦИЯ «ВНИИЭМ»,3
261,Возмещение части затрат ЗАО «ПАРТНЕР Ф» на уплату лизинг...,ЗАО «ПАРТНЕР Ф»,1


In [13]:
start = clean["Начало действия договора субсидии"]
end = clean["Конец действия договора субсидии"]

print("Диапазон начала действия договора:", start.min(), "-", start.max())
print("Диапазон окончания действия договора:", end.min(), "-", end.max())
print("Договоров, где конец раньше начала:", (end < start).sum())
print("Договоров, где конец совпадает с началом:", (end == start).sum())
print("\nСрок действия договора, лет:")
print((end - start).value_counts().sort_index())

Диапазон начала действия договора: 2012 - 2017
Диапазон окончания действия договора: 2014 - 2025
Договоров, где конец раньше начала: 0
Договоров, где конец совпадает с началом: 0

Срок действия договора, лет:
1      37
2      29
3     177
4      24
6       2
10      1
Name: count, dtype: int64


In [14]:
print("Число договоров по году начала:")
print(start.value_counts().sort_index())

Число договоров по году начала:
Начало действия договора субсидии
2012    24
2013    31
2014    35
2015    55
2016    57
2017    68
Name: count, dtype: int64


### 3.5. Неявные дубликаты получателей

Одно и то же предприятие записано по-разному: полная организационно-правовая форма,
сокращение, форма с указанием «города Москвы». Формально это разные строки, фактически —
один получатель. Оцениваем масштаб, огрубленно снимая ОПФ и кавычки.

In [15]:
OPF = (
    r"^(Открытое акционерное общество|Закрытое акционерное общество"
    r"|Публичное акционерное общество|Акционерное общество"
    r"|Общество с ограниченной ответственностью"
    r"|Казенное предприятие города Москвы|Казенное предприятие"
    r"|Государственное унитарное предприятие"
    r"|ОАО|ООО|ЗАО|ПАО|АО|КП|ГУП)\s*"
)

recipients = clean["Получатель субсидии"]
core = recipients.str.replace(OPF, "", regex=True).str.replace(r"[«»\"]", "", regex=True).str.upper()

print("Строк в таблице:", len(clean))
print("Уникальных получателей как есть:", recipients.nunique())
print("Уникальных получателей после снятия ОПФ:", core.nunique())

Строк в таблице: 270
Уникальных получателей как есть: 234
Уникальных получателей после снятия ОПФ: 206


In [16]:
collisions = (
    pd.DataFrame({"нормализовано": core, "как в данных": recipients})
    .groupby("нормализовано")["как в данных"]
    .agg(["nunique", lambda s: " | ".join(sorted(s.unique()))])
    .rename(columns={"nunique": "вариантов написания", "<lambda_0>": "варианты"})
)
collisions[collisions["вариантов написания"] > 1].sort_values(
    "вариантов написания", ascending=False
).head(10)

,вариантов написания,варианты
нормализовано,,
ЭПИЭЛ,3,АО «Эпиэл» | Акционерное общество «Эпиэл» | Закрытое акц...
КОРПОРАЦИЯ РАЗВИТИЯ ЗЕЛЕНОГРАДА,3,КП «Корпорация развития Зеленограда» | Казенное предприя...
ЦАРИЦЫНО,3,ОАО «ЦАРИЦЫНО» | ОАО «Царицыно» | Открытое акционерное о...
ЛЕД-ЭФФЕКТ,3,ООО «ЛЕД-ЭФФЕКТ» | ООО «ЛЕД-Эффект» | Общество с огранич...
МИКОЯНОВСКИЙ МЯСОКОМБИНАТ,3,ЗАО «МИКОЯНОВСКИЙ МЯСОКОМБИНАТ» | ЗАО «Микояновский мясо...
ДЫМОВСКОЕ КОЛБАСНОЕ ПРОИЗВОДСТВО,3,ООО «ДЫМОВСКОЕ КОЛБАСНОЕ ПРОИЗВОДСТВО» | Общество с огра...
НАУЧНО-ПРОИЗВОДСТВЕННОЕ ПРЕДПРИЯТИЕ ЦЕНТР ПЕРСПЕКТИВНЫХ ТЕХНОЛОГИЙ,2,ООО Научно-производственное предприятие «Центр перспекти...
ЩЕРБИНСКИЙ ЛИФТОСТРОИТЕЛЬНЫЙ ЗАВОД,2,ОАО «Щербинский лифтостроительный завод» | Открытое акци...
ЦЕНТР КОНСАЛТИНГА В ИННОВАЦИОННОЙ СФЕРЕ,2,ООО «Центр консалтинга в инновационной сфере» | Общество...


In [17]:
print("Топ получателей по числу договоров (без нормализации):")
print(recipients.value_counts().head(8))

Топ получателей по числу договоров (без нормализации):
Получатель субсидии
Казенное предприятие города Москвы «Корпорация развития Зеленограда»           6
Открытое акционерное общество «Царицыно»                                       5
КП «Корпорация развития Зеленограда»                                           4
ЗАО «МИКОЯНОВСКИЙ МЯСОКОМБИНАТ»                                                3
Общество с ограниченной ответственностью «ДЫМОВСКОЕ КОЛБАСНОЕ ПРОИЗВОДСТВО»    3
Общество с ограниченной ответственностью «Дымовское колбасное производство»    3
Казенное предприятие «Корпорация развития Зеленограда»                         3
Общество с ограниченной ответственностью «Альфа Автоматив Техноложиз»          2
Name: count, dtype: int64


## 4. Профилирующий отчет

Отчет строится по подготовленной таблице `clean`: служебная строка снята, типы приведены,
лишние пробелы убраны. Без этой подготовки профайлер увидел бы семь текстовых столбцов
и не построил бы ни распределения сумм, ни корреляций.

In [18]:
profile = ProfileReport(
    clean,
    title="Субсидии организациям инновационной и промышленной сфер (данные Москвы, набор 591)",
    explorative=True,
    progress_bar=False,
    dataset={
        "description": (
            "Перечень субсидий, предоставленных организациям, осуществляющим деятельность "
            "в инновационной и промышленной сферах. Портал открытых данных Правительства "
            "Москвы, набор 591, выгрузка от 17.04.2018."
        ),
        "url": "https://data.mos.ru/opendata/591",
    },
)
profile.to_file(REPORT)
print("Отчет сохранен:", REPORT)

  0%|          | 0/7 [00:00<?, ?it/s]

100%|██████████| 7/7 [00:00<00:00, 248.85it/s]

Отчет сохранен: /Users/iamartempn/Education/HSE/courses/module0/Homework1/report.html


## 5. Что показал анализ

Краткая сводка находок; развернутые выводы — в файле `conclusions.docx`.

In [19]:
summary = pd.Series(
    {
        "строк / столбцов": f"{clean.shape[0]} / {clean.shape[1]}",
        "пропусков в данных": int(clean.isna().sum().sum()),
        "полных дубликатов строк": int(clean.duplicated().sum()),
        "значений с переносом строки (до очистки)": int(
            df[text_cols].apply(lambda s: s.astype(str).str.contains("\n")).sum().sum()
        ),
        "получателей как есть / после снятия ОПФ": f"{recipients.nunique()} / {core.nunique()}",
        "минимальная сумма субсидии, руб.": int(summa.min()),
        "медианная сумма субсидии, руб.": int(summa.median()),
        "максимальная сумма субсидии, руб.": int(summa.max()),
        "период действия договоров, годы": f"{start.min()}-{end.max()}",
    },
    name="значение",
)
summary.to_frame()

,значение
строк / столбцов,270 / 7
пропусков в данных,0
полных дубликатов строк,0
значений с переносом строки (до очистки),83
получателей как есть / после снятия ОПФ,234 / 206
"минимальная сумма субсидии, руб.",1
"медианная сумма субсидии, руб.",7000000
"максимальная сумма субсидии, руб.",865000000
"период действия договоров, годы",2012-2025
